In [ ]:
# ==================== 1. 설치 ====================
!pip install -q langchain langchain-ollama langchain-community langchain-chroma pypdf gradio beautifulsoup4

# zstd 설치 (Ollama 설치의 전제 조건)
!apt-get update && apt-get install -y zstd

# Ollama 설치
!curl -fsSL https://ollama.com/install.sh | sh

# ==================== 2. Ollama 서버 실행 (백그라운드) ====================
import subprocess
import time

# Ollama 서버 시작
subprocess.Popen(["ollama", "serve"])
time.sleep(5)  # 서버가 뜰 때까지 대기

# 모델 다운로드 (처음 한 번만)
!ollama pull gemma4:e4b
!ollama pull llama3.2
!ollama pull nomic-embed-text

print("✅ Ollama 준비 완료!")

In [ ]:
# ==================== 2. Google Drive 마운트 ====================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==================== 3. 설정 ====================
# ←←← 여기를 본인 폴더 경로로 수정하세요 ←←←
FOLDER_PATH = "/content/drive/MyDrive/fintech_internship/강의 자료/네이버페이"   # 예: MyDrive/RAG_Docs

In [ ]:
# ==================== 4. Markdown 파일 로드 ====================
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

# Markdown 파일만 로드
loader = DirectoryLoader(
    FOLDER_PATH,
    glob="**/*.md",                    # .md 파일만 불러옴
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

docs = loader.load()
print(f"✅ 불러온 Markdown 파일 수: {len(docs)}개")

In [ ]:
# ==================== 5. 문서 분할 & 벡터 DB ====================
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200
)
splits = text_splitter.split_documents(docs)

embeddings = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="./chroma_db_md"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

In [ ]:
# ==================== 6. LLM & Chain ====================
llm = ChatOllama(model="gemma4:e4b", temperature=0.3)

template = """너는 네이버페이의 AI 어시스턴트야. 고객이 내용을 쉽게 이해하도록 돕고, 질문에 대해 정확하고 친절하게 답변해.
모든 답변은 실제 문서에 근거해야 하며, 추측하여 대답하지 마.
고객이 이해하기 어려운 용어가 포함된 경우, 괄호 안에 쉬운 설명을 덧붙여 줘.
아래 Context(내 자료)를 참고하여 질문에 답변해줘.
Context에 없는 내용은 솔직하게 "제공된 자료에 해당 정보가 없습니다."라고 답변해.

Context:
{context}

Question: {question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# ==================== 7. Gradio 챗봇 ====================
import subprocess
import time
import requests

# 초기 인사말 + 가이드 질문 목록
GREETING_MESSAGE = """안녕하세요! 네이버페이 이용 도우미입니다. 궁금하신 질문을 선택해주세요.

📌 **자주 묻는 질문 목록**

**💳 결제 연동**
1. 네이버페이 온라인 결제 연동은 어떻게 시작하나요?
2. 단건 결제와 자동 결제(정기결제)의 차이점이 무엇인가요?
3. 결제창은 어떻게 호출하나요?
4. 결제 승인 절차가 어떻게 되나요?

**🔄 자동 결제**
5. 자동결제를 등록하는 방법을 알려주세요.
6. 자동결제 등록을 해지하려면 어떻게 하나요?
7. 자동결제 예약 및 승인은 어떻게 처리하나요?

**↩️ 취소 / 환불**
8. 단건 결제를 취소하는 방법이 궁금해요.
9. 자동결제 취소는 어떻게 하나요?

**📊 정산 / 내역 조회**
10. 결제 내역을 조회하는 방법을 알려주세요.
11. 일별 정산 내역은 어떻게 조회하나요?
12. 건별 정산 내역 조회 방법이 궁금해요.
13. 수수료 상세 내역은 어떻게 확인하나요?

**🧾 기타**
14. 현금영수증 금액은 어떻게 조회하나요?
15. 네이버 포인트 적립 요청은 어떻게 하나요?
16. 지원하는 카드 / 은행 정보가 궁금해요.
17. API 인증 방식은 어떻게 되나요?
18. 방화벽 설정 시 허용해야 할 IP가 무엇인가요?

위 번호를 입력하거나, 직접 질문을 입력해 주세요! 😊"""

# 번호 입력 시 실제 질문으로 변환
GUIDE_QUESTIONS = {
    "1": "네이버페이 온라인 결제 연동은 어떻게 시작하나요?",
    "2": "단건 결제와 자동 결제(정기결제)의 차이점이 무엇인가요?",
    "3": "결제창은 어떻게 호출하나요?",
    "4": "결제 승인 절차가 어떻게 되나요?",
    "5": "자동결제를 등록하는 방법을 알려주세요.",
    "6": "자동결제 등록을 해지하려면 어떻게 하나요?",
    "7": "자동결제 예약 및 승인은 어떻게 처리하나요?",
    "8": "단건 결제를 취소하는 방법이 궁금해요.",
    "9": "자동결제 취소는 어떻게 하나요?",
    "10": "결제 내역을 조회하는 방법을 알려주세요.",
    "11": "일별 정산 내역은 어떻게 조회하나요?",
    "12": "건별 정산 내역 조회 방법이 궁금해요.",
    "13": "수수료 상세 내역은 어떻게 확인하나요?",
    "14": "현금영수증 금액은 어떻게 조회하나요?",
    "15": "네이버 포인트 적립 요청은 어떻게 하나요?",
    "16": "지원하는 카드 / 은행 정보가 궁금해요.",
    "17": "API 인증 방식은 어떻게 되나요?",
    "18": "방화벽 설정 시 허용해야 할 IP가 무엇인가요?",
}

# Ollama 서버가 꺼져 있으면 자동으로 재시작
def ensure_ollama_running():
    try:
        requests.get("http://localhost:11434/api/tags", timeout=3)
    except Exception:
        print("⚠️ Ollama 서버가 꺼져 있습니다. 재시작 중...")
        subprocess.Popen(["ollama", "serve"])
        time.sleep(5)
        print("✅ Ollama 서버 재시작 완료")

# type='messages' 포맷: {"role": "user"|"assistant", "content": "..."}
initial_history = [{"role": "assistant", "content": GREETING_MESSAGE}]

with gr.Blocks(title="네이버페이 이용 도우미") as demo:
    gr.Markdown("# 💚 네이버페이 이용 도우미")
    gr.Markdown(f"총 {len(docs)}개의 공식 문서를 기반으로 답변합니다.")

    chatbot = gr.Chatbot(
        value=initial_history,
        label="네이버페이 도우미",
        height=600,
        type="messages",           # openai 스타일 dict 포맷 (deprecation 해결)
        buttons=["copy"],          # show_copy_button 대체
        allow_tags=False,
    )

    with gr.Row():
        msg = gr.Textbox(
            placeholder="질문 번호(1~18) 또는 직접 질문을 입력하세요...",
            label="질문 입력",
            scale=9,
        )
        send_btn = gr.Button("전송", variant="primary", scale=1)

    # 1단계: 사용자 메시지를 즉시 화면에 표시하고 입력창 초기화
    def add_user_message(message, history):
        if not message.strip():
            return history, message
        history = history + [
            {"role": "user", "content": message},
            {"role": "assistant", "content": None},  # 봇 응답 자리 예약
        ]
        return history, ""

    # 2단계: Ollama 상태 확인 후 LLM 응답을 받아 채워넣기
    def add_bot_response(history):
        ensure_ollama_running()
        # 마지막 사용자 메시지 찾기
        user_message = next(
            (m["content"] for m in reversed(history) if m["role"] == "user"),
            ""
        )
        actual_question = GUIDE_QUESTIONS.get(user_message.strip(), user_message)
        try:
            bot_response = rag_chain.invoke(actual_question)
        except Exception as e:
            bot_response = f"오류가 발생했습니다. 잠시 후 다시 시도해주세요.\n\n_(오류: {e})_"
        history[-1]["content"] = bot_response
        return history

    send_btn.click(
        add_user_message, inputs=[msg, chatbot], outputs=[chatbot, msg]
    ).then(
        add_bot_response, inputs=chatbot, outputs=chatbot
    )
    msg.submit(
        add_user_message, inputs=[msg, chatbot], outputs=[chatbot, msg]
    ).then(
        add_bot_response, inputs=chatbot, outputs=chatbot
    )

demo.launch(share=True, debug=True)